# Complete Dash Tutorial

Dash is a productive Python framework for building web applications. It's built on top of:
- **Flask** for the web server
- **React.js** for the front-end
- **Plotly.js** for interactive graphs

With Dash, you can create interactive dashboards, data apps, and visualization tools using only Python.

**Key features:**
- Pure Python – no JavaScript required
- Highly customizable layouts
- Reactive callbacks for interactivity
- Enterprise-ready (authentication, deployment)
- Open source with commercial support

This tutorial covers everything from basic app structure to advanced callbacks and deployment.

## Installation

```bash
pip install dash
```

For additional components:
```bash
pip install dash-core-components dash-html-components dash-table
```

For Bootstrap themes (optional but recommended):
```bash
pip install dash-bootstrap-components
```

In [ ]:
# Core imports
import dash
from dash import dcc, html, Input, Output, State, callback, ctx
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# For bootstrap styling
import dash_bootstrap_components as dbc

# For data (use built-in datasets)
tips = px.data.tips()
iris = px.data.iris()

print("Libraries imported successfully!")
print(f"Dash version: {dash.__version__}")

## 1. Your First Dash App

A Dash app consists of:
- **Layout**: describes what the app looks like (HTML + Dash components)
- **Callbacks**: define how the app responds to user interactions

Note: The following cells create mini-apps. To run them, copy the code into a `.py` file and run with `python app.py`, or use `app.run_server(debug=True)` in a script. In Jupyter, you can use `jupyter-dash` (extra install). We'll show the code structure here.

In [ ]:
# Minimal app (this is a blueprint – to run, save as app.py)
"""
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import plotly.express as px
import pandas as pd

app = dash.Dash(__name__)

# Load data
df = px.data.iris()

# Layout
app.layout = html.Div([
    html.H1('Iris Data Explorer', style={'textAlign': 'center'}),
    dcc.Dropdown(
        id='species-dropdown',
        options=[{'label': s, 'value': s} for s in df['species'].unique()],
        value='setosa'
    ),
    dcc.Graph(id='scatter-plot')
])

# Callback
@app.callback(
    Output('scatter-plot', 'figure'),
    Input('species-dropdown', 'value')
)
def update_plot(selected_species):
    filtered_df = df[df['species'] == selected_species]
    fig = px.scatter(filtered_df, x='sepal_length', y='sepal_width',
                     title=f'Sepal Dimensions for {selected_species}')
    return fig

if __name__ == '__main__':
    app.run_server(debug=True)
"""
print("Copy the code above into a file named app.py and run it to see the interactive dashboard.")

## 2. Layout Components

Dash provides several component libraries:
- `dash.html` – HTML elements (`Div`, `H1`, `Button`, `Img`, etc.)
- `dash.dcc` – Core components (`Dropdown`, `Graph`, `Slider`, `Input`, `Checklist`, `DatePicker`, etc.)
- `dash.dbc` – Bootstrap components (better styling)
- `dash.dt` – DataTable (interactive tables)

In [ ]:
# Example layout combining multiple components
layout_example = html.Div([
    # Header
    html.H1('Dashboard Title', className='header'),
    
    # Row 1: Dropdown and Slider
    html.Div([
        html.Label('Select Category:'),
        dcc.Dropdown(
            id='demo-dropdown',
            options=[{'label': 'Option A', 'value': 'A'},
                     {'label': 'Option B', 'value': 'B'}],
            value='A',
            clearable=False,
            style={'width': '50%'}
        ),
        html.Br(),
        html.Label('Range Slider:'),
        dcc.RangeSlider(
            id='range-slider',
            min=0, max=100, step=1,
            value=[20, 80],
            marks={i: str(i) for i in range(0, 101, 20)}
        ),
    ]),
    
    # Row 2: Graph
    html.Div([
        dcc.Graph(id='main-graph', figure={})
    ]),
    
    # Row 3: Input and Output text
    html.Div([
        dcc.Input(id='user-input', type='text', placeholder='Type something...'),
        html.Div(id='output-text')
    ]),
    
    # Row 4: Checkbox and Radio items
    html.Div([
        dcc.Checklist(
            id='checklist',
            options=[{'label': 'Show mean', 'value': 'mean'},
                     {'label': 'Show median', 'value': 'median'}],
            value=['mean']
        ),
        dcc.RadioItems(
            id='radio',
            options=[{'label': 'Linear', 'value': 'linear'},
                     {'label': 'Log', 'value': 'log'}],
            value='linear'
        )
    ]),
    
    # Hidden div to store intermediate data
    dcc.Store(id='store-data', data={})
])

print("Layout structure created with many core components.")

## 3. Callbacks – Making the App Interactive

Callbacks connect inputs (user interactions) to outputs (component updates).

In [ ]:
# Basic callback: input → output
"""
@app.callback(
    Output('output-div', 'children'),
    Input('input-box', 'value')
)
def update_output(value):
    return f'You typed: {value}'
"""
print("A callback function is triggered whenever an Input property changes.")

In [ ]:
# Multiple inputs
"""
@app.callback(
    Output('result', 'children'),
    Input('dropdown', 'value'),
    Input('slider', 'value')
)
def combine_inputs(dropdown_val, slider_val):
    return f'Selected: {dropdown_val}, Value: {slider_val}'
"""
print("Multiple Inputs: callback fires when any of the inputs change.")

In [ ]:
# Multiple outputs (Dash 2.0+)
"""
@app.callback(
    Output('graph1', 'figure'),
    Output('graph2', 'figure'),
    Input('dropdown', 'value')
)
def update_graphs(selected):
    fig1 = px.scatter(...)
    fig2 = px.bar(...)
    return fig1, fig2
"""
print("You can return a tuple of outputs in the same order as the Output decorator.")

In [ ]:
# Using State (prevents firing on page load)
"""
@app.callback(
    Output('output', 'children'),
    Input('submit-button', 'n_clicks'),
    State('input-field', 'value')
)
def update_on_click(n_clicks, value):
    if n_clicks is None:
        return 'Click the button'
    return f'Submitted: {value}'
"""
print("State properties are passed to the callback but do NOT trigger it.")

In [ ]:
# Prevent initial callback (with `prevent_initial_call=True`)
"""
@app.callback(
    Output('output', 'children'),
    Input('button', 'n_clicks'),
    prevent_initial_call=True
)
def update_on_click(n_clicks):
    return f'Clicked {n_clicks} times'
"""
print("Use `prevent_initial_call=True` to avoid running the callback when the app loads.")

## 4. Advanced Callback Patterns

- **Pattern‑matching callbacks** – dynamic component IDs
- **Callback context** – which input triggered the callback
- **Clientside callbacks** – JavaScript for performance
- **Long callbacks** – background tasks (Dash Enterprise)

In [ ]:
# Using callback context to know which input triggered
"""
from dash import ctx

@app.callback(
    Output('result', 'children'),
    Input('button1', 'n_clicks'),
    Input('button2', 'n_clicks')
)
def which_button(btn1, btn2):
    if ctx.triggered_id == 'button1':
        return 'Button 1 was clicked'
    elif ctx.triggered_id == 'button2':
        return 'Button 2 was clicked'
    return 'No button clicked yet'
"""
print("dash.callback_context.triggered_id tells you which input fired.")

In [ ]:
# Pattern‑matching callbacks (for dynamic lists of components)
"""
from dash import ALL, MATCH

# Create dynamic buttons
@app.callback(
    Output({'type': 'output', 'index': MATCH}, 'children'),
    Input({'type': 'button', 'index': MATCH}, 'n_clicks')
)
def update_output(click):
    return f'Clicked {click} times'
"""
print("Pattern‑matching callbacks use dictionaries with 'type' and 'index' to match dynamic components.")

## 5. Styling with Dash Bootstrap Components (DBC)

Dash Bootstrap Components (DBC) makes it easy to create responsive, professional layouts using Bootstrap 5.

In [ ]:
# Example DBC layout
"""
import dash_bootstrap_components as dbc

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

app.layout = dbc.Container([
    dbc.Row([
        dbc.Col(html.H1('Dashboard'), width=12)
    ]),
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardHeader('Controls'),
                dbc.CardBody([
                    dcc.Dropdown(id='dropdown', options=[...])
                ])
            ])
        ], width=3),
        dbc.Col([
            dcc.Graph(id='graph')
        ], width=9)
    ])
])
"""
print("DBC provides `Container`, `Row`, `Col`, `Card`, `Button`, `Alert`, `Modal`, etc.")

# Available themes: BOOTSTRAP, DARKLY, FLATLY, LUMEN, SLATE, etc.
themes = ['BOOTSTRAP', 'DARKLY', 'FLATLY', 'LUMEN', 'SLATE']
print(f"Some DBC themes: {themes}")

## 6. Interactive Graphs – Advanced

Plotly graphs in Dash can have **clickData**, **hoverData**, **selectedData**, **relayoutData** (zoom/pan).

In [ ]:
# Reacting to clicks on a graph
"""
@app.callback(
    Output('click-output', 'children'),
    Input('scatter-plot', 'clickData')
)
def display_click_data(clickData):
    if clickData is None:
        return 'Click on a point'
    point = clickData['points'][0]
    return f'Clicked point: x={point["x"]}, y={point["y"]}'
"""
print("Properties you can use: `clickData`, `hoverData`, `selectedData`, `relayoutData`.")

In [ ]:
# Reacting to zoom/pan (relayoutData)
"""
@app.callback(
    Output('range-output', 'children'),
    Input('graph', 'relayoutData')
)
def display_zoom(relayoutData):
    if relayoutData and 'xaxis.range[0]' in relayoutData:
        x0 = relayoutData['xaxis.range[0]']
        x1 = relayoutData['xaxis.range[1]']
        return f'X range: [{x0:.2f}, {x1:.2f}]'
    return 'Zoom or pan the graph'
"""
print("`relayoutData` captures zoom, pan, and axis range changes.")

## 7. DataTable – Interactive Tables

`dash_table.DataTable` provides editable, sortable, filterable tables.

In [ ]:
import dash_table

df_sample = px.data.tips().head(10)

table = dash_table.DataTable(
    id='table',
    columns=[{'name': col, 'id': col} for col in df_sample.columns],
    data=df_sample.to_dict('records'),
    editable=True,
    filter_action='native',
    sort_action='native',
    page_action='native',
    page_current=0,
    page_size=5,
    style_table={'overflowX': 'auto'},
    style_cell={'textAlign': 'left'},
    style_header={'backgroundColor': 'lightgray', 'fontWeight': 'bold'}
)
print("DataTable with filtering, sorting, pagination, editing.")

## 8. Multi‑Page Apps

Use `dash.dcc.Location` and `dash.dcc.Link` to create multiple pages without reloading the entire app.

In [ ]:
# Structure for a multi‑page Dash app
"""
app = dash.Dash(__name__, suppress_callback_exceptions=True)

app.layout = html.Div([
    dcc.Location(id='url', refresh=False),
    html.Div([
        dcc.Link('Home', href='/'),
        ' | ',
        dcc.Link('Page 1', href='/page1'),
        ' | ',
        dcc.Link('Page 2', href='/page2'),
    ]),
    html.Div(id='page-content')
])

@app.callback(
    Output('page-content', 'children'),
    Input('url', 'pathname')
)
def display_page(pathname):
    if pathname == '/page1':
        return html.Div('This is page 1')
    elif pathname == '/page2':
        return html.Div('This is page 2')
    else:
        return html.Div('Welcome to the home page')
"""
print("Multi‑page apps use `dcc.Location` to manage URL routing.")

## 9. Authentication & Deployment

- **Authentication**: Use Dash Enterprise (paid) or integrate Flask-Login / OAuth
- **Deployment**:
  - **Dash Enterprise** – commercial hosting
  - **Heroku**, **PythonAnywhere**, **AWS EC2**, **GCP App Engine**
  - **Docker** container
  - **Render**, **DigitalOcean**

In [ ]:
# Example gunicorn command for production
# gunicorn app:server

print("For production, use a WSGI server like gunicorn or waitress.")
print("Make sure to set `debug=False` and host='0.0.0.0'.")

## 10. Complete Example: Interactive Dashboard

Here's a fully functional dashboard with multiple components.

In [ ]:
# Complete dashboard code – copy into app.py and run
"""
import dash
from dash import dcc, html, Input, Output
import plotly.express as px
import pandas as pd

app = dash.Dash(__name__)

# Load data
tips = px.data.tips()

app.layout = html.Div([
    html.H1('Tip Explorer Dashboard', style={'textAlign': 'center'}),
    
    html.Div([
        html.Label('Select Sex:'),
        dcc.Dropdown(
            id='sex-dropdown',
            options=[{'label': 'All', 'value': 'All'}] + [{'label': s, 'value': s} for s in tips['sex'].unique()],
            value='All'
        ),
        
        html.Label('Select Day:'),
        dcc.Dropdown(
            id='day-dropdown',
            options=[{'label': 'All', 'value': 'All'}] + [{'label': d, 'value': d} for d in tips['day'].unique()],
            value='All'
        ),
        
        html.Label('Total Bill Range:'),
        dcc.RangeSlider(
            id='bill-slider',
            min=0, max=50, step=1,
            marks={i: str(i) for i in range(0, 51, 10)},
            value=[0, 50]
        ),
    ], style={'padding': '20px'}),
    
    dcc.Graph(id='scatter-plot'),
    dcc.Graph(id='bar-plot')
])

@app.callback(
    [Output('scatter-plot', 'figure'),
     Output('bar-plot', 'figure')],
    [Input('sex-dropdown', 'value'),
     Input('day-dropdown', 'value'),
     Input('bill-slider', 'value')]
)
def update_plots(sex, day, bill_range):
    df = tips.copy()
    if sex != 'All':
        df = df[df['sex'] == sex]
    if day != 'All':
        df = df[df['day'] == day]
    df = df[(df['total_bill'] >= bill_range[0]) & (df['total_bill'] <= bill_range[1])]
    
    scatter = px.scatter(df, x='total_bill', y='tip', color='sex', size='size',
                         title='Total Bill vs Tip')
    bar = px.bar(df.groupby('day')['tip'].mean().reset_index(),
                 x='day', y='tip', title='Average Tip by Day')
    return scatter, bar

if __name__ == '__main__':
    app.run_server(debug=True)
"""
print("Copy the above code into a Python file (e.g., dashboard.py) and run it.")

## 11. Best Practices

1. **Use callbacks with `prevent_initial_call=True`** to avoid unnecessary computations.
2. **Store large data in `dcc.Store`** to avoid passing it back and forth.
3. **Use `dash.callback_context`** to optimise which inputs trigger updates.
4. **Organise callbacks into separate files** for larger apps.
5. **Use Dash Bootstrap Components** for responsive layouts.
6. **Set `debug=False` in production**.
7. **Use `gunicorn` or `waitress`** as the WSGI server.
8. **Cache expensive computations** with `flask_caching`.

## 12. Further Resources

- [Official Dash Documentation](https://dash.plotly.com/)
- [Dash Bootstrap Components](https://dash-bootstrap-components.opensource.faculty.ai/)
- [Dash Enterprise](https://plotly.com/dash/)
- [Awesome Dash](https://github.com/ucg8j/awesome-dash)
- [Dash Sample Apps](https://dash.plotly.com/sample-apps)

## Summary

You've learned:
- **Installation and basic app structure**
- **Layout components** (html, dcc, dbc, datatable)
- **Callbacks** – inputs, outputs, state, context, multiple outputs
- **Advanced callbacks** – pattern‑matching, clientside, long callbacks
- **Styling with Dash Bootstrap Components**
- **Interactive graphs** – clickData, hoverData, relayoutData
- **DataTable** – editable, sortable, filterable
- **Multi‑page apps** using `dcc.Location`
- **Authentication and deployment**
- **Complete example dashboard**
- **Best practices**

Dash empowers Python developers to create powerful, interactive web applications without writing JavaScript.